# SocialEngineerArena GRPO Training

Use this notebook on Hugging Face/Colab compute after credits are available. It is wired around the live OpenEnv reward function, not a static label-only dataset.

In [3]:
# If running from notebooks/, move to repo root first.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

!pip install -q -U pip
!pip install -q openenv-core trl transformers accelerate datasets
!pip install -q -e .

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from scripts.train_grpo_placeholder import format_prompt, MODEL_NAME
from social_engineer_arena.models import ArenaAction
from social_engineer_arena.server.environment import SocialEngineerArenaEnvironment

env = SocialEngineerArenaEnvironment()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample_obs = env.reset()
sample_prompt = format_prompt(sample_obs)
print("Model:", MODEL_NAME)
print("Role:", sample_obs.role, "Scenario:", sample_obs.scenario_id)
print(sample_prompt[:700])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

C:\Users\vinod\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vinod\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

# Local rollout sanity check: generate actions, score in env, track rewards.
def generate_action_json(prompt: str, max_new_tokens: int = 220) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    # Keep only assistant completion portion if prompt is echoed.
    return text[len(prompt):].strip() if text.startswith(prompt) else text


def parse_action(completion: str) -> ArenaAction:
    try:
        # Try full JSON first.
        return ArenaAction.model_validate_json(completion)
    except Exception:
        # Fallback: extract first JSON object in text.
        start = completion.find("{")
        end = completion.rfind("}")
        if start != -1 and end != -1 and end > start:
            return ArenaAction.model_validate_json(completion[start:end + 1])
        return ArenaAction(explanation=completion)


rewards = []
rows = []
for i in range(8):
    obs = env.reset()
    prompt = format_prompt(obs)
    completion = generate_action_json(prompt)
    action = parse_action(completion)
    _, reward, done = env.step(action)
    rewards.append(reward)
    rows.append({
        "i": i,
        "role": obs.role,
        "scenario_id": obs.scenario_id,
        "reward": reward,
        "done": done,
        "verdict": action.verdict,
    })

rows, sum(rewards) / len(rewards)

In [ ]:
# Optional: lightweight training placeholder loop (supervised-style warmup prompt collection)
# This notebook now verifies end-to-end env scoring locally.
# For full RL (GRPO), use TRL with the same reward path: env.reset() -> model completion -> ArenaAction -> env.step().

print("Local verification complete if rewards are non-trivial and vary across scenarios.")
print("Next step: plug this reward pipeline into GRPOTrainer on GPU compute.")